# Segmentation model 2

## Улучшения относительно baseline

### Loss функция
Используется **комбинированная функция потерь**:
$$
loss = 0.25 \cdot BCEWithLogitsLoss + 0.75 \cdot DiceLoss
$$
→ Значительно лучше сходится, чем чистый Dice.

### Аугментации
- Geometric: HorizontalFlip, VerticalFlip, RandomRotate90, ShiftScaleRotate
- Color: RandomBrightnessContrast, HueSaturationValue, **CLAHE** (борьба с бликами)
- Noise & Regularization: GaussNoise, CoarseDropout, **GridDropout** (GridMask)
- Mix-аугментация: **CutMix** (на уровне батча, работает с масками)

### Архитектура модели
- **Модель**: `UnetPlusPlus` (одна из самых мощных для бинарной сегментации)
- **Энкодер**: `efficientnet-b3` (ImageNet pretrained)
- Разрешение: **512×512**

### Обучение и регуляризация
- SWA (Stochastic Weight Averaging) + обновление BN
- Finetune на полном датасете (5 эпох с низким LR)
- Автоматический поиск лучшего threshold на валидации
- Оптимизатор: AdamW + CosineAnnealingLR

### Оптимизации скорости и памяти
- AMP (Automatic Mixed Precision)
- `cudnn.benchmark = True`, TF32 matmul, high precision matmul
- DataLoader: `prefetch_factor`, `persistent_workers`, `pin_memory`
- CutMix через кастомный `collate_fn`

### Инференс
- **TTA**: 6 видов (оригинал + H-flip + V-flip + Rot90 + Rot270 + Diagonal flip)
- **Scale TTA**: 0.75× и 1.25× (включено)
- Ресайз предсказания обратно на оригинальный размер изображения
- Сериализация маски в JSON-формат (как требует соревнование)

### Дополнительно
- Полный бэкап всех чекпоинтов и submission на **Google Drive**

# Segmentation model 2

## Init

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
import os
from google.colab import userdata

# Получаем учетные данные из менеджера секретов Colab
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Устанавливаем переменные окружения для Kaggle
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key
os.environ['KAGGLEHUB_CACHE'] = '/'

# Создаем директорию .kaggle, если ее нет
!mkdir -p ~/.kaggle

# Создаем файл kaggle.json с учетными данными
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# Устанавливаем правильные права доступа для файла kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# Теперь вы можете использовать kaggle CLI или kagglehub.login()
# kagglehub.login() не всегда нужен, если kaggle.json настроен правильно
print("Kaggle API credentials configured successfully!")

Kaggle API credentials configured successfully!


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dl_lab_3_product_segmentation_path = kagglehub.competition_download('dl-lab-3-product-segmentation')
# packagemanager_pm_113281741_at_03_23_2026_18_04_38_path = kagglehub.notebook_output_download('packagemanager/pm-113281741-at-03-23-2026-18-04-38')

print('Data source import complete.')


100%|██████████| 191M/191M [00:03<00:00, 62.9MB/s]

Extracting files...


Data source import complete.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Путь к папке на Google Drive
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/segmentation_checkpoints


In [ ]:
pip install segmentation-models-pytorch timm opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.8 MB/s eta 0:00:00


In [ ]:
pip uninstall -y torch torchvision

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128


In [ ]:
pip install --no-cache-dir torch==2.7.0 torchvision==0.22.0 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 76.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 267.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 182.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 237.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 243.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 214.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 151.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 178.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 MB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 301.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.5/

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

## Preparation / train

### GPU speed optimization

In [ ]:
import torch

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print(" CUDA optimizations enabled:")
print(" cudnn.benchmark = True")
print(" TF32 matmul enabled")
print(" high precision matmul")

 CUDA optimizations enabled:
 cudnn.benchmark = True
 TF32 matmul enabled
 high precision matmul


### CONFIG

In [ ]:
DATA_ROOT = Path(
    r"/competitions/dl-lab-3-product-segmentation/train"
)
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
SAVE_DIR = Path("./seg_train_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/segmentation_checkpoints")
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Флаг: копировать ли на Drive (можно отключить для быстрых тестов)
BACKUP_TO_DRIVE = True

IMG_SIZE = 512
BATCH_SIZE = 12 # тут смотреть по GPU mem
NUM_EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_RATIO = 0.15

# === SPEED UPDATES ===
NUM_WORKERS = 8
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True

SEED = 42
THRESHOLD = 0.5

MODEL_NAME = "UnetPlusPlus"
ENCODER_NAME = "efficientnet-b3" # или efficientnet-b0 для экономии времени
ENCODER_WEIGHTS = "imagenet"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import shutil

def backup_checkpoint_to_drive(local_path: Path, drive_path: Path, fold: int, epoch: int, is_best: bool = False):
    if not BACKUP_TO_DRIVE or not is_best:
        return

    try:
        drive_fold_dir = drive_path / f"fold_{fold}"
        drive_fold_dir.mkdir(parents=True, exist_ok=True)

        drive_file = drive_fold_dir / "best.pth"
        shutil.copy2(local_path, drive_file)

        meta_file = drive_fold_dir / f"checkpoint_metadata.txt"
        with open(meta_file, 'a') as f:
            f.write(f"Epoch {epoch}: BEST - {local_path.name}\n")

        print(f"  Backed up BEST model to Drive: {drive_file}")
    except Exception as e:
        print(f"  Failed to backup to Drive: {e}")

### Utils

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def dice_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)

    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


seed_everything(SEED)


### Loss (нейронка говорит, что если добавить BCE, будет лучше сходиться)

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

dice_loss = smp.losses.DiceLoss(
    mode=smp.losses.BINARY_MODE,
    from_logits=True
)

def combined_loss(logits, targets):
    bce = bce_loss(logits, targets)
    dice = dice_loss(logits, targets)
    return 0.25 * bce + 0.75 * dice

### Augmentations

In [ ]:
import albumentations as A
import random
import numpy as np

def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            scale=(0.9, 1.1),
            translate_percent=(-0.05, 0.05),
            rotate=(-20, 20),
            p=0.5
        ),
        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(p=0.3),
        A.CLAHE(p=0.3),
        A.GaussNoise(p=0.2),
        A.CoarseDropout(
            max_holes=8,
            hole_height_range=(1, img_size // 20),
            hole_width_range=(1, img_size // 20),
            p=0.3
        ),
        A.GridDropout(
            ratio=0.4,
            unit_size_min=img_size // 32,
            unit_size_max=img_size // 8,
            holes_number_x=3,
            holes_number_y=3,
            random_offset=True,
            fill_value=0,
            p=0.35
        ),
    ], p=1.0)


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
    ])


def cutmix_collate_fn(batch, alpha=1.0, p=0.1):
    images, masks = zip(*batch)
    images = torch.stack(images)  # [B, 3, H, W]
    masks  = torch.stack(masks)   # [B, 1, H, W]

    if random.random() > p or len(images) < 2:
        return images, masks

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0)).to(images.device)

    W, H = images.shape[3], images.shape[2]
    cut_ratio = np.sqrt(1. - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)

    cx = np.random.randint(cut_w // 2, W - cut_w // 2)
    cy = np.random.randint(cut_h // 2, H - cut_h // 2)

    bbx1 = max(0, cx - cut_w // 2)
    bby1 = max(0, cy - cut_h // 2)
    bbx2 = min(W, cx + cut_w // 2)
    bby2 = min(H, cy + cut_h // 2)

    images[:, :, bby1:bby2, bbx1:bbx2] = images[rand_index, :, bby1:bby2, bbx1:bbx2]
    masks[:, :,  bby1:bby2, bbx1:bbx2] = masks[rand_index,  :, bby1:bby2, bbx1:bbx2]

    return images, masks

### Dataset

In [ ]:
class BinarySegDataset(Dataset):
    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int = 352,
        encoder_name: str = "resnet34",
        encoder_weights: str | None = "imagenet",
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.img_size = img_size
        self.transforms = transforms

        self.preprocess_input = None
        if encoder_weights is not None:
            self.preprocess_input = get_preprocessing_fn(
                encoder_name, pretrained=encoder_weights
            )

        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(
                f"No paired image/mask samples found in {self.images_dir} and {self.masks_dir}"
            )

        print(f"Found {len(self.samples)} paired samples")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise RuntimeError(f"Cannot read image: {image_path}")

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Cannot read mask: {mask_path}")

        if self.transforms is not None:
            augmented = self.transforms(image=image_rgb, mask=mask)
            image_rgb = augmented["image"]
            mask = augmented["mask"]
        else:
            image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size))
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image_rgb = image_rgb.astype(np.float32)

        if self.preprocess_input is not None:
            image_rgb = self.preprocess_input(image_rgb)
        else:
            image_rgb = image_rgb / 255.0

        mask = (mask > 0).astype(np.float32)

        image = torch.from_numpy(image_rgb.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask[None, ...]).float()

        return image, mask

### Model

In [ ]:
def build_model() -> nn.Module:
    if MODEL_NAME == "Unet":
        model = smp.Unet(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif MODEL_NAME == "FPN":
        model = smp.FPN(
            encoder_name=ENCODER_NAME,
            encoder_weights=ENCODER_WEIGHTS,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}")
    return model


### TRAIN / VAL LOOPS

In [ ]:
from tqdm import tqdm


scaler = torch.cuda.amp.GradScaler()

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Training", leave=False)


    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits.detach(), masks)
        running_iou += iou_score_from_logits(logits.detach(), masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0

    pbar = tqdm(loader, desc="Validation", leave=False)


    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks)
        running_iou += iou_score_from_logits(logits, masks)

    n = len(loader)
    return {
        "loss": running_loss / n,
        "dice": running_dice / n,
        "iou": running_iou / n,
    }

/tmp/ipykernel_2904/669458370.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


### Split

In [ ]:
from sklearn.model_selection import train_test_split

full_dataset = BinarySegDataset(
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    img_size=IMG_SIZE,
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    transforms=None,
)

indices = list(range(len(full_dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=VAL_RATIO,
    random_state=SEED
)

train_dataset = torch.utils.data.Subset(
    BinarySegDataset(
        IMAGES_DIR,
        MASKS_DIR,
        IMG_SIZE,
        ENCODER_NAME,
        ENCODER_WEIGHTS,
        transforms=get_train_transforms(IMG_SIZE),
    ),
    train_idx,
)

val_dataset = torch.utils.data.Subset(
    BinarySegDataset(
        IMAGES_DIR,
        MASKS_DIR,
        IMG_SIZE,
        ENCODER_NAME,
        ENCODER_WEIGHTS,
        transforms=get_val_transforms(IMG_SIZE),
    ),
    val_idx,
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

### Loader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=PERSISTENT_WORKERS,
    collate_fn=lambda batch: cutmix_collate_fn(batch, alpha=1.0, p=0.1)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR,
    persistent_workers=PERSISTENT_WORKERS
)

### Model + Optimizer + SWA

In [ ]:
model = build_model().to(DEVICE)

loss_fn = combined_loss

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

swa_model = AveragedModel(model)
swa_start = int(NUM_EPOCHS * 0.7)

swa_scheduler = SWALR(optimizer, swa_lr=1e-5)

### Training loop

In [ ]:
best_val_dice = -1.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, loss_fn, DEVICE)
    val_metrics = validate_one_epoch(model, val_loader, loss_fn, DEVICE)

    if epoch > swa_start:
        swa_model.update_parameters(model)
        swa_scheduler.step()
        current_lr = swa_scheduler.get_last_lr()[0]
    else:
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": train_metrics["loss"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"train_dice={row['train_dice']:.4f} | "
        f"val_dice={row['val_dice']:.4f}"
    )

    if row["val_dice"] > best_val_dice:
        best_val_dice = row["val_dice"]

        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {
                "MODEL_NAME": MODEL_NAME,
                "ENCODER_NAME": ENCODER_NAME,
                "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
                "IMG_SIZE": IMG_SIZE,
            },
        }, SAVE_DIR / "best.pth")

        backup_checkpoint_to_drive(
          SAVE_DIR / "best.pth",
          DRIVE_CHECKPOINT_DIR,
          fold=0,
          epoch=epoch,
          is_best=True
        )

        print(f"Saved BEST model: {best_val_dice:.4f}")

### SWA BN update

In [ ]:
update_bn(train_loader, swa_model, device=DEVICE)

torch.save({
    "model_state_dict": swa_model.state_dict(),
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "ENCODER_NAME": ENCODER_NAME,
        "ENCODER_WEIGHTS": ENCODER_WEIGHTS,
        "IMG_SIZE": IMG_SIZE,
    },
}, SAVE_DIR / "swa.pth")
print(" Saved SWA model (after BN update)")

### THRESHOLD TUNING

In [ ]:
import numpy as np
import torch

@torch.no_grad()
def evaluate_thresholds(model, loader, device):
    model.eval()

    thresholds = np.arange(0.3, 0.7, 0.05)
    results = []

    for t in thresholds:
        dice_scores = []

        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            probs = torch.sigmoid(logits)

            preds = (probs > t).float()

            preds = preds.view(preds.size(0), -1)
            masks_flat = masks.view(masks.size(0), -1)

            intersection = (preds * masks_flat).sum(dim=1)
            denom = preds.sum(dim=1) + masks_flat.sum(dim=1)

            dice = (2.0 * intersection + 1e-7) / (denom + 1e-7)
            dice_scores.append(dice.mean().item())

        mean_dice = np.mean(dice_scores)
        results.append((t, mean_dice))

        print(f"Threshold {t:.2f} -> Dice {mean_dice:.4f}")

    best_t, best_score = max(results, key=lambda x: x[1])

    print("\n=== BEST THRESHOLD ===")
    print(f"Threshold: {best_t:.2f}")
    print(f"Dice: {best_score:.4f}")

    global THRESHOLD
    THRESHOLD = best_t

    return best_t


best_threshold = evaluate_thresholds(swa_model, val_loader, DEVICE)

THRESHOLD = best_threshold

print(f"\nUsing THRESHOLD = {THRESHOLD:.2f} for inference")

### Finetune on FULL DATA

In [ ]:
ckpt = torch.load(SAVE_DIR / "swa.pth", map_location=DEVICE)

model = build_model().to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

full_dataset = BinarySegDataset(
    IMAGES_DIR,
    MASKS_DIR,
    IMG_SIZE,
    ENCODER_NAME,
    ENCODER_WEIGHTS,
    transforms=get_train_transforms(IMG_SIZE),
)

full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=lambda batch: cutmix_collate_fn(batch, alpha=1.0, p=0.05)  # чуть ниже p на финтюне
)


for epoch in range(1, 6):
    metrics = train_one_epoch(model, full_loader, optimizer, loss_fn, DEVICE)
    scheduler.step()
    print(f"Finetune epoch {epoch}/5 | dice={metrics['dice']:.4f} | loss={metrics['loss']:.4f}")

torch.save({
    "model_state_dict": model.state_dict(),
    "config": ckpt["config"],
}, SAVE_DIR / "final.pth")
print(" Saved FINAL model (SWA + finetune)")

## Submission formation

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn


In [ ]:
def load_checkpoint_from_drive(fold: int, checkpoint_type: str = "best"):
    """
    Загружает чекпоинт с Google Drive
    checkpoint_type: "best" или "last"
    """
    drive_fold_dir = DRIVE_CHECKPOINT_DIR / f"fold_{fold}"
    checkpoint_path = drive_fold_dir / f"{checkpoint_type}.pth"

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    print(f"Loading from Drive: {checkpoint_path}")
    return torch.load(checkpoint_path, map_location="cpu")

### Config

In [ ]:
TEST_IMAGES_DIR = Path(
    r"/competitions/dl-lab-3-product-segmentation/test_images"
)
OUTPUT_CSV = "submission.csv"

# путь к вашему чекпоинту после обучения
# CHECKPOINT_PATH = Path(r"./seg_train_runs/best.pth")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
# THRESHOLD = 0.5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### HELPERS

In [ ]:
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    if data.size == 0:
        return None
    return cv2.imdecode(data, flags)


def build_model(model_name: str, encoder_name: str, encoder_weights=None):
    if model_name == "Unet":
        model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "UnetPlusPlus":
        model = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    elif model_name == "FPN":
        model = smp.FPN(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
        )
    else:
        raise ValueError(f"Unsupported MODEL_NAME: {model_name}")
    return model




def serialize_mask(mask2d: np.ndarray) -> str:
    return json.dumps(mask2d.astype(np.uint8).tolist(), separators=(",", ":"))


### Load checkpoint



In [ ]:
ckpt = torch.load(SAVE_DIR / "final.pth", map_location=DEVICE)

model = build_model(
    model_name=ckpt["config"]["MODEL_NAME"],
    encoder_name=ckpt["config"]["ENCODER_NAME"],
    encoder_weights=None,
)

model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE)
model.eval()

models = [model]

Loading: seg_train_runs/fold_0/best.pth
   → Compiling model fold 0 (inference speedup)
Loading: seg_train_runs/fold_1/best.pth
   → Compiling model fold 1 (inference speedup)
Loading: seg_train_runs/fold_2/best.pth
   → Compiling model fold 2 (inference speedup)
Loading: seg_train_runs/fold_3/best.pth
   → Compiling model fold 3 (inference speedup)
Loading: seg_train_runs/fold_4/best.pth
   → Compiling model fold 4 (inference speedup)


### INFERENCE

In [ ]:
import torch.nn.functional as F


use_scale = True


preprocess_input = get_preprocessing_fn(
    ckpt["config"]["ENCODER_NAME"],
    pretrained=ckpt["config"]["ENCODER_WEIGHTS"]
)

image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])

if not image_paths:
    raise FileNotFoundError(f"No images found in: {TEST_IMAGES_DIR}")

print(f"Found {len(image_paths)} test images")

rows = []

img_size = IMG_SIZE

with torch.no_grad():
    for i, img_path in enumerate(image_paths, 1):
        img_bgr = cv2_imread_unicode(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            print(f"[skip] cannot read: {img_path}")
            continue

        H, W = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        inp = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        inp = inp.astype(np.float32)

        if preprocess_input is not None:
            inp = preprocess_input(inp)
        else:
            inp = inp / 255.0

        inp = torch.from_numpy(inp.transpose(2, 0, 1)).float().unsqueeze(0).to(DEVICE)

        preds = []
        for model in models:
            tta_preds = []
            # original
            logits = model(inp)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            tta_preds.append(probs)
            # horizontal flip
            inp_flip = torch.flip(inp, dims=[3])
            logits = model(inp_flip)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.flip(probs, axis=1)
            tta_preds.append(probs)
            # vertical flip
            inp_flip = torch.flip(inp, dims=[2])
            logits = model(inp_flip)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.flip(probs, axis=0)
            tta_preds.append(probs)
            # rotate 90
            inp_rot = torch.rot90(inp, k=1, dims=[2,3])
            logits = model(inp_rot)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.rot90(probs, k=3)
            tta_preds.append(probs)
            # rotate 270
            inp_rot = torch.rot90(inp, k=3, dims=[2,3])
            logits = model(inp_rot)
            probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
            probs = np.rot90(probs, k=1)
            tta_preds.append(probs)
            # Diagonal flip (транспонирование)
            inp_diag = inp.transpose(2, 3)
            pred_diag = torch.sigmoid(model(inp_diag))[0, 0].cpu().numpy()
            tta_preds.append(pred_diag.transpose(1, 0))
            # Масштабирование (опционально)
            if use_scale:
                for scale in [0.75, 1.25]:
                    new_size = int(img_size * scale)
                    inp_scaled = F.interpolate(inp, size=(new_size, new_size), mode='bilinear')
                    pred_scaled = torch.sigmoid(model(inp_scaled))[0, 0].cpu().numpy()
                    pred_scaled = cv2.resize(pred_scaled, (img_size, img_size))
                    tta_preds.append(pred_scaled)

            tta_pred = np.mean(tta_preds, axis=0)
            preds.append(tta_pred)

        pred = np.mean(preds, axis=0)
        if pred.shape != (H, W):
            pred = cv2.resize(pred.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)

        mask = (pred > THRESHOLD).astype(np.uint8)
        rows.append({"ImageId": img_path.name, "mask": serialize_mask(mask)})

        if i % 100 == 0 or i == len(image_paths):
            print(f"Processed {i}/{len(image_paths)}")

submission_df = pd.DataFrame(rows)
submission_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

print(f"\nSaved local submission to: {OUTPUT_CSV}")

if BACKUP_TO_DRIVE:
    try:
        drive_submission_path = DRIVE_CHECKPOINT_DIR / "submission.csv"
        shutil.copy2(OUTPUT_CSV, drive_submission_path)
        print(f"Backed up submission to Drive: {drive_submission_path}")
    except Exception as e:
        print(f"Failed to backup submission to Drive: {e}")

display(submission_df.head())

Found 2000 test images
Processed 100/2000
Processed 200/2000
Processed 300/2000
Processed 400/2000
Processed 500/2000
Processed 600/2000
Processed 700/2000
Processed 800/2000
Processed 900/2000
Processed 1000/2000
Processed 1100/2000
Processed 1200/2000
Processed 1300/2000
Processed 1400/2000
Processed 1500/2000
Processed 1600/2000
Processed 1700/2000
Processed 1800/2000
Processed 1900/2000
Processed 2000/2000

Saved local submission to: submission.csv
Backed up submission to Drive: /content/drive/MyDrive/segmentation_checkpoints/submission.csv


,ImageId,mask
0,10.107.215.111_20260119142748_a0be018c-d498-4b...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
1,10.107.215.111_20260119203335_58a6ef5c-c360-4e...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
2,10.107.215.111_20260119203341_bb74ac11-3f14-4a...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
3,10.107.215.111_20260122115937_db9b4af5-0f1e-42...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
4,10.107.215.111_20260124135952_6fd575ec-50e8-4f...,"[[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,..."
